# Module 13 - Lab: Generate New Insights from Reused Data

In this lab you reuse a stored measurement for a question it was not originally collected to answer. The notebook follows four steps from the lecture: **import, inspect, vary, and record/store**.

Local `metadata.json` is used only to locate a matching RO-Crate ZIP. After import, the crate metadata replaces it as the authoritative description of the reused dataset:

- **Drivetrain illuminance:** average the detected bright phases and explore whether the recorded light level indicates potentially poor working-light conditions.
- **Suspension acceleration:** integrate longitudinal and lateral motion to estimate travelled distance, heading, and a local 2D route from start to end.

Both analyses are exploratory. A pattern becomes a tentative hypothesis, not automatically a confirmed finding.

## Learning goals

- Reuse data and metadata beyond their original purpose.
- State provenance, assumptions, analytical choices, and limitations.
- Vary one influential parameter and compare the result.
- Distinguish an indicator from a causal conclusion.
- Store the new artefact together with the parameters needed to reproduce it.


## Section 1: Launch and Import

Run the prepared import cell. It reuses the robust data loading and metadata-driven setup from Lab 6 and adds only the new Module 13 analyses.


In [ ]:
# Section 1: Import Libraries
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import (
    load_metadata_context,
    summarize_metadata_context,
)
from data_format_loader import (
    analysis_config_table,
    create_time_quality_report,
    load_recorded_data,
    prepare_measurement_analysis,
    summarize_loaded_data,
)
from ro_crate_loader import (
    apply_ro_crate_to_metadata,
    find_latest_ro_crate,
    load_ro_crate,
    summarize_ro_crate,
)
from module13_analysis import (
    analyze_bright_phase_working_conditions,
    calculate_suspension_route,
    compare_bright_phase_thresholds,
    compare_route_deadbands,
    display_module13_story,
    plot_bright_phase_threshold_comparison,
    plot_bright_phase_working_conditions,
    plot_route_deadband_comparison,
    plot_suspension_route,
)
from reused_metadata import (
    apply_reuse_analysis_metadata,
    build_reused_metadata,
    get_reuse_analysis_metadata,
    sha256_file,
    write_reused_metadata,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 4)

print('Libraries imported successfully.')


## Section 2: Import a Stored Run and Its Metadata

`metadata.json` determines the generated RO-Crate name and the matching import selection; it does not store an RO-Crate path. Inside the archive, `ro-crate-metadata.json` describes the run, `mainEntity` selects the measurement file, and `hasPart` links the packaged data entities. The examples follow the [RO-Crate 1.3 specification](https://www.researchobject.org/ro-crate/specification/1.3/) and use the same shared exporter intended for Lab 10. Each ZIP is generated from the current top-level selection in `metadata.json`; its embedded `metadata/metadata.json` contains only the selected analysis and measurement-mode metadata. Detailed recording metadata can still be extracted from the workbook or packaged `data/meta/` folder after import.

Before analysing, inspect the crate's provenance, licence, format, measured quantity, and declared main data file.


In [ ]:
# Section 2: Local Analysis Metadata
metadata_context = load_metadata_context(project_root)
metadata = metadata_context['public_metadata']

print('Local metadata and analysis configuration:')
display(pd.json_normalize(summarize_metadata_context(metadata_context), sep='.').T.rename(columns={0: 'value'}))


### Select the RO-Crate

The default RO-Crate is selected from `output/ro-crates/` using the active metadata and use case. If several matching exports exist, the newest dated ZIP is used. To reuse the other stored run, change only `ro_crate_path_override`. The crate—not the file name—declares the measurement type, quantity, version, and primary data entity.


In [ ]:
# Section 2: Import the Selected RO-Crate
ro_crate_path_override = None
# ro_crate_path_override = 'output/ro-crates/2026-07-16_drivetrain_illuminance_example_raw_v0-1-0.ro-crate.zip'
# ro_crate_path_override = 'output/ro-crates/2026-07-16_suspension_acceleration_example_raw_v0-1-0.ro-crate.zip'

selected_ro_crate_path = (
    project_root / ro_crate_path_override
    if ro_crate_path_override
    else find_latest_ro_crate(metadata, project_root)
)
ro_crate_context = load_ro_crate(selected_ro_crate_path, project_root)
ro_crate_summary = summarize_ro_crate(ro_crate_context)
metadata = apply_ro_crate_to_metadata(metadata, ro_crate_context, project_root)
selected_data_path = ro_crate_context['main_data_path']
crate_data_reference = f"{ro_crate_summary['archive_path']}!/{ro_crate_context['main_entity_id']}"

print('Imported RO-Crate ZIP:')
display(pd.DataFrame([ro_crate_summary]).T.rename(columns={0: 'value'}))

print('\nResolved main data file:', selected_data_path)
print('Analysis key from RO-Crate:', f"{metadata['measurement_type']}_{metadata['quantity']}")


In [ ]:
# Section 2: Load the Reused Measurement
loaded_recorded_data = load_recorded_data(selected_data_path, project_root)
df_raw = loaded_recorded_data['table']
recorded_data_metadata = {
    'recorded_data_path': crate_data_reference,
    'detected_format': loaded_recorded_data['format'],
    'extracted_metadata': loaded_recorded_data['recording_metadata'],
}

print('Loaded file:', selected_data_path.name)
print('Rows, columns:', df_raw.shape)
loaded_data_summary = summarize_loaded_data(loaded_recorded_data)
loaded_data_summary['path'] = crate_data_reference
display(pd.DataFrame([loaded_data_summary]))
display(df_raw.head())


### Observation 1: Provenance and Reuse

Document the reused source before exploring it.

- Who generated or published the original data according to the RO-Crate?
- What was its original purpose?
- Which data entity is declared as `mainEntity`?
- Which metadata, licence, and format information are available?
- Why might this dataset support the new question?

- 


## Section 3: Inspect Before Computing

Check shape, columns, units, ranges, missing values, duplicates, and time steps. Reuse becomes unreliable when two datasets or coordinate systems only look compatible.


In [ ]:
# Section 3: Inspect Data Structure and Quality
print('Column names:')
print(df_raw.columns.tolist())

print('\nData types:')
print(df_raw.dtypes)

print('\nMissing values per column:')
display(df_raw.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', df_raw.duplicated().sum())

print('\nSummary statistics:')
display(df_raw.describe(include='all').T)


In [ ]:
# Section 3: Add Parameters for the New Reuse Question
analysis_key = f"{metadata['measurement_type']}_{metadata['quantity']}"
if analysis_key == 'drivetrain_illuminance':
    reuse_parameter_overrides = {
        'minimum_bright_phase_mean_lx': 500.0,
        'bright_phase_min_duration_s': 0.3,
        'bright_phase_thresholds_to_compare_lx': [300.0, 500.0, 1000.0],
    }
elif analysis_key == 'suspension_acceleration':
    reuse_parameter_overrides = {
        'route_initial_heading_deg': 0.0,
        'route_end_speed_m_per_s': 0.0,
        'route_apply_linear_speed_drift_correction': True,
        'route_clip_negative_speed': True,
        'route_min_speed_m_per_s': 0.5,
        'route_lateral_deadband_m_per_s2': 0.05,
        'route_max_yaw_rate_deg_per_s': 90.0,
        'route_deadbands_to_compare_m_per_s2': [0.0, 0.05, 0.1, 0.2],
    }
else:
    raise ValueError(f'No Module 13 reuse parameters exist for {analysis_key!r}.')

reuse_analysis_metadata = get_reuse_analysis_metadata(analysis_key, reuse_parameter_overrides)
metadata = apply_reuse_analysis_metadata(metadata, analysis_key, reuse_analysis_metadata)
display(pd.DataFrame([reuse_analysis_metadata]).T.rename(columns={0: 'Module 13 value'}))

# Prepare source preprocessing plus the separate reuse parameters.
analysis_context = prepare_measurement_analysis(df_raw, metadata)
analysis_config = analysis_context['config']
df_analysis = analysis_context['df_analysis']
time_column = analysis_context['time_column']
value_column = analysis_context['value_column']

print('Analysis mode:', analysis_context['analysis_key'])
display(analysis_config_table(analysis_context))
display(create_time_quality_report(df_analysis, time_column))


### Observation 2: Fitness for the New Purpose

- Are the units and axes sufficient for the new calculation?
- Which important contextual variables are missing?
- Would you combine this run with another group's run? Why or why not?

- 


## Section 4: State the New Question and Assumptions

The table below makes the exploratory reasoning explicit. Do not hide these assumptions when communicating the result.


In [ ]:
# Section 4: Exploratory Analysis Story
module13_story = display_module13_story(analysis_context)
display(module13_story)


## Section 5: Explore a New Insight

The cell adapts to the active measurement mode.

**Drivetrain:** Bright phases are detected relative to the signal itself, then their illuminance values are averaged. The default comparison value is 500 lx, based on the German [ASR A3.4 workplace-light guidance](https://www.baua.de/DE/Themen/Arbeitsgestaltung/Gefaehrdungsbeurteilung/Handbuch-Gefaehrdungsbeurteilung/Expertenwissen/Arbeitsumgebungsbedingungen/Beleuchtung-Licht/Beleuchtung-Licht_dossier) example for writing, reading, and data-processing workplaces. This is a workplace-light comparison—not a threshold that diagnoses fatigue. Sensor position and orientation may also differ from the actual workplace plane.

**Suspension:** Forward acceleration is integrated to speed and distance. The turn estimate uses `lateral acceleration = speed × yaw rate`; heading and position are then integrated. The result uses local coordinates: the start is `(0, 0)` and the initial heading points along positive x. It is not a GPS track.


In [ ]:
# Section 5: Run the Mode-Specific Reuse Analysis
reuse_result = None
route_result = None

if analysis_context['analysis_key'] == 'drivetrain_illuminance':
    reuse_result = analyze_bright_phase_working_conditions(analysis_context, metadata)
    display(reuse_result['summary'])
    display(reuse_result['phases'])
    plot_bright_phase_working_conditions(reuse_result)
elif analysis_context['analysis_key'] == 'suspension_acceleration':
    route_result = calculate_suspension_route(analysis_context)
    display(route_result['summary'])
    plot_suspension_route(route_result)
else:
    raise ValueError(f"No Module 13 analysis is configured for {analysis_context['analysis_key']!r}.")


### Observation 3: First Exploratory Pattern

For drivetrain, describe whether the **mean of the bright phases** exceeds the selected threshold. Phrase the outcome as an indicator of measured lighting conditions, not evidence of tired people.

For suspension, report estimated route distance, straight-line displacement, and the end position relative to the start. Describe whether the route shape is physically plausible.

- 


## Section 6: Vary One Parameter

Parameters are assumptions. This section shows how one analytical choice changes the conclusion.

- **Drivetrain:** vary the minimum illuminance threshold.
- **Suspension:** vary the lateral-acceleration deadband. A larger deadband ignores more small sideways accelerations when estimating turns.


In [ ]:
# Section 6: Parameter Sensitivity
threshold_comparison = None
route_parameter_comparison = None
route_comparison_results = None

if analysis_context['analysis_key'] == 'drivetrain_illuminance':
    thresholds_lx = analysis_config.get('bright_phase_thresholds_to_compare_lx', [300, 500, 1000])
    threshold_comparison = compare_bright_phase_thresholds(reuse_result, thresholds_lx)
    display(threshold_comparison)
    plot_bright_phase_threshold_comparison(threshold_comparison)
else:
    deadbands = analysis_config.get('route_deadbands_to_compare_m_per_s2', [0.0, 0.05, 0.1, 0.2])
    route_parameter_comparison, route_comparison_results = compare_route_deadbands(analysis_context, deadbands)
    display(route_parameter_comparison)
    plot_route_deadband_comparison(route_comparison_results)


### Observation 4: Parameter Effects

- Which result changes when the parameter changes?
- Which part remains stable?
- Which parameter value would you report, and why?
- What independent measurement would best validate the exploratory result?

- 


## Section 7: Record a Tentative Finding

Keep exploration and confirmation separate. Complete the text variables below so the reasoning is stored in `metadata_reused.json`, not only displayed in the notebook.


In [ ]:
# Section 7: Text that will be stored with the reused analysis
research_question = (
    'Do the bright-phase light levels indicate potentially poor working-light conditions?'
    if analysis_key == 'drivetrain_illuminance'
    else 'What travelled distance and local 2D route can be estimated from the acceleration data?'
)
assumptions = ''
analytical_choices = ''
limitations = ''
tentative_finding = ''
future_hypothesis = ''

print('Research question:', research_question)
print('Complete the remaining text variables before storing the artefact.')


## Section 8: Store the New Artefact

The final cell stores the summary and detail tables together with `metadata_reused.json`. The new metadata file records the source RO-Crate, its checksum, the original preprocessing parameters, the separate Module 13 parameters, the tentative result, and all generated artefacts. It deliberately excludes unrelated measurement modes.


In [ ]:
# Section 8: Write the reused artefact and its separate metadata
safe_dataset_name = selected_data_path.stem.replace(' ', '_')
output_prefix = f'lab13_{metadata.get("measurement_type", "measurement")}_{safe_dataset_name}'
output_dir = project_root / 'outputs' / output_prefix
output_dir.mkdir(parents=True, exist_ok=True)
summary_output_path = output_dir / 'summary.csv'
reused_metadata_path = output_dir / 'metadata_reused.json'

if reuse_result is not None:
    new_insight_summary = reuse_result['summary']
    detail_table = reuse_result['phases']
    detail_output_path = output_dir / 'bright_phases.csv'
    parameter_table = threshold_comparison
else:
    new_insight_summary = route_result['summary']
    route_columns = [
        time_column,
        'route_speed_m_per_s',
        'route_distance_m',
        'route_heading_deg',
        'route_x_m',
        'route_y_m',
    ]
    detail_table = route_result['route'][route_columns]
    detail_output_path = output_dir / 'route_points.csv'
    parameter_table = route_parameter_comparison

new_insight_summary.to_csv(summary_output_path, index=False)
detail_table.to_csv(detail_output_path, index=False)

source_preprocessing_parameters = ro_crate_context['embedded_metadata']['analysis'][analysis_key]
reused_metadata = build_reused_metadata(
    analysis_key=analysis_key,
    source_ro_crate={
        'path': ro_crate_summary['archive_path'],
        'sha256': sha256_file(ro_crate_context['archive_path']),
        'conforms_to': ro_crate_summary['conforms_to'],
        'date_published': ro_crate_summary['date_published'],
        'main_data_entity': ro_crate_context['main_entity_id'],
        'embedded_metadata_entity': ro_crate_context['embedded_metadata_id'],
    },
    source_dataset={
        'measurement_type': metadata['measurement_type'],
        'quantity': metadata['quantity'],
        'run_name': metadata['run_name'],
        'data_stage': metadata['data_stage'],
        'version': metadata['version'],
        'data_reference': crate_data_reference,
        'detected_format': recorded_data_metadata['detected_format'],
    },
    source_preprocessing_parameters=source_preprocessing_parameters,
    lab13_parameters=reuse_analysis_metadata,
    result_summary=new_insight_summary.to_dict(orient='records'),
    parameter_comparison=parameter_table.to_dict(orient='records'),
    artifacts={
        'summary_csv': summary_output_path.name,
        'detail_csv': detail_output_path.name,
    },
    question=research_question,
    assumptions=assumptions,
    analytical_choices=analytical_choices,
    limitations=limitations,
    tentative_finding=tentative_finding,
    future_hypothesis=future_hypothesis,
)
write_reused_metadata(reused_metadata_path, reused_metadata)

print('Wrote summary:', summary_output_path)
print('Wrote detailed artefact:', detail_output_path)
print('Wrote reuse metadata:', reused_metadata_path)
